# VIS4AI · Notebook 01 — Setup and first run

Goal of this notebook: get everything working and **let both models solve the same GSM8K math problems**.
No metrics or plots yet — just a clean setup and the raw answers, saved to Google Drive.

| Model | Role |
|---|---|
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | main model: a small *reasoning* model that "thinks" in `<think> … </think>` before answering |
| `Qwen/Qwen2.5-Math-1.5B` | baseline: the math base model that R1-Distill was fine-tuned from (no long reasoning) |

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (allow Google Drive access when asked)
3. **Disconnected?** Just *Run all* again. Answers are saved after every batch, and problems that are already done are skipped.

## 1. Setup
Install packages, check the GPU and connect Google Drive (where all results are stored).

*You may see a warning "You are sending unauthenticated requests to the HF Hub". It is harmless: the models and data are public.*

In [ ]:
!pip install -q "transformers>=4.56" "accelerate>=0.33" "datasets>=2.20"

In [ ]:
import gc, json, math, os, re, time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList, set_seed

assert torch.cuda.is_available(), "No GPU! Go to Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Settings
All settings are in this one cell. Each model is run the way its authors recommend, so the settings differ per model:

| | R1-Distill | Qwen2.5-Math |
|---|---|---|
| prompt | no system prompt, instruction in the user message (DeepSeek) | instruction as system prompt (Qwen's official math format) |
| decoding | sampling, temperature 0.6, top-p 0.95 (DeepSeek; greedy can make it loop) | greedy (Qwen) |
| `max_new_tokens` | 4096 — its thinking is long; with 2048, 1 in 5 answers was cut off | 1024 — answers are short |
| `stop_after_answer` | no — the model ends by itself after `</think>` + answer | **yes** — as a base model it does not stop by itself and starts inventing new questions after its answer, so we stop it as soon as it has written a `\boxed{...}` answer |
| answer = | the last `\boxed{...}` **after** `</think>` (`thinks=True`). No `</think>` = still thinking when the token limit hit = no answer | the first non-empty `\boxed{...}` (later boxes would belong to invented questions) |

* `N_PROBLEMS`: how many GSM8K problems to try. Raising it later only runs the new problems.
* `RUN_NAME`: results are saved in a folder with this name. **Keep it to resume a run; change it to start a fresh run** (e.g. after changing settings), so old and new answers are never mixed.

In [ ]:
MODELS = {
    "r1-distill": dict(name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", thinks=True,
                       system_prompt=False, sampling=True,  max_new_tokens=4096, stop_after_answer=False),
    "qwen-math":  dict(name="Qwen/Qwen2.5-Math-1.5B", thinks=False,
                       system_prompt=True,  sampling=False, max_new_tokens=1024, stop_after_answer=True),
}

N_PROBLEMS  = 50       # number of GSM8K test problems
BATCH_SIZE  = 16       # problems generated at the same time (16 fits on a T4)
TEMPERATURE = 0.6      # only used by models with sampling=True
TOP_P       = 0.95
SEED        = 42

RUN_NAME    = "run2"   # change to start a fresh run; keep to resume
RESULTS_DIR = Path("/content/drive/MyDrive/VIS4AI/results") / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Results are saved in:", RESULTS_DIR)

## 2. Load the dataset
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) test split: 1,319 grade-school math word problems. The correct answer is the number after `####` in the solution text.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

problems = [
    {"id": i,
     "question": gsm8k[i]["question"],
     "gold": gsm8k[i]["answer"].split("####")[-1].strip().replace(",", "")}
    for i in range(N_PROBLEMS)
]

print(f"{len(problems)} problems loaded. Example:\n")
print(problems[0]["question"])
print("\nCorrect answer:", problems[0]["gold"])

## 3. Helper functions
* **`build_prompt`** — the question plus *"Please reason step by step, and put your final answer within \boxed{}."*, as user message or system prompt depending on the model. The tokenizer's chat template adds the model-specific special tokens.
* **`extract_answer`** — gets the answer out of the output, following the "answer =" row of the settings table. Empty `\boxed{}` (a repeated instruction) is ignored. `None` means no answer.
* **`load_results` / `save_results`** — read/append a `.jsonl` file on Drive (one line per problem), which makes the run resumable.
* **`load_model` / `unload_model`** — load a model in float16 (the T4 has no bfloat16) and free the GPU afterwards.
* **`generate_answers`** — let the model answer a batch of questions. Also records how many tokens it wrote and whether it **finished** (`False` = it hit the token limit and was cut off).

In [ ]:
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

def build_prompt(tokenizer, question, system_prompt):
    if system_prompt:
        messages = [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": question}]
    else:
        messages = [{"role": "user", "content": question + "\n" + INSTRUCTION}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def boxed_answers(text):
    # Contents of all non-empty \boxed{...} in the text, in order (handles nested braces like \boxed{\frac{1}{2}}).
    answers = []
    for match in re.finditer(r"\\boxed\{", text):
        depth = 1
        for j in range(match.end(), len(text)):
            depth += {"{": 1, "}": -1}.get(text[j], 0)
            if depth == 0:
                if text[match.end():j].strip():
                    answers.append(text[match.end():j].strip())
                break
    return answers


def extract_answer(text, thinks):
    if thinks:
        if "</think>" not in text:
            return None                       # still thinking when the token limit was reached
        boxes = boxed_answers(text.split("</think>")[-1])
        return boxes[-1] if boxes else None
    boxes = boxed_answers(text)
    return boxes[0] if boxes else None


def to_number(answer):
    # '$1,234' -> 1234.0, '20\%' -> 20.0, '\frac{3}{4}' -> 0.75, None if no number.
    if answer is None:
        return None
    a = re.sub(r"\\[dt]?frac\{(\d+)\}\{(\d+)\}", r"\1/\2", answer).replace(",", "")
    if m := re.fullmatch(r"\s*(-?\d+)/(\d+)\s*", a):
        return int(m.group(1)) / int(m.group(2))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", a)
    return float(numbers[-1]) if numbers else None


def is_correct(answer, gold):
    value = to_number(answer)
    return value is not None and math.isclose(value, float(gold), abs_tol=1e-6)


# Quick sanity check of the answer parsing
assert extract_answer("thinking... </think> so \\boxed{18}.", thinks=True) == "18"
assert extract_answer("thinking \\boxed{5} but ran out of tokens", thinks=True) is None
assert extract_answer("\\boxed{7}. Please put your answer within \\boxed{}. \\boxed{3}", thinks=False) == "7"
assert extract_answer("no box here", thinks=False) is None
assert is_correct("\\$1,234", "1234") and is_correct("20\\%", "20") and is_correct("75.00", "75")
assert to_number("\\frac{3}{4}") == 0.75 and not is_correct(None, "5")
print("Answer parsing works.")

In [ ]:
def load_results(path):
    if not path.exists():
        return []
    results = []
    for line in path.read_text().splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            pass    # a half-written line from a disconnect is ignored (that problem is simply redone)
    return results


def save_results(path, results):
    with open(path, "a") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())    # make sure it really reaches Google Drive


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"           # needed for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda").eval()
    print(f"Loaded {model_name} | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, tokenizer


def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


class StopAfterAnswer(StoppingCriteria):
    # Stops each sequence as soon as it contains a complete, non-empty \boxed{...} answer.
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer, self.prompt_length = tokenizer, prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        texts = self.tokenizer.batch_decode(input_ids[:, self.prompt_length:], skip_special_tokens=True)
        return torch.tensor([bool(boxed_answers(t)) for t in texts], device=input_ids.device)


@torch.no_grad()
def generate_answers(model, tokenizer, questions, cfg):
    prompts = [build_prompt(tokenizer, q, cfg["system_prompt"]) for q in questions]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    prompt_length = inputs["input_ids"].shape[1]

    # Stop at the normal end token, and also at <|im_end|> (end of a chat turn for Qwen models)
    stop_ids = [tokenizer.eos_token_id]
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end != tokenizer.unk_token_id:
        stop_ids.append(im_end)

    if cfg["sampling"]:
        decoding = dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
    else:
        decoding = dict(do_sample=False, temperature=None, top_p=None, top_k=None)   # greedy
    outputs = model.generate(
        **inputs, **decoding,
        max_new_tokens=cfg["max_new_tokens"],
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=stop_ids,
        stopping_criteria=StoppingCriteriaList([StopAfterAnswer(tokenizer, prompt_length)]) if cfg["stop_after_answer"] else None,
    )

    results = []
    for tokens in outputs[:, prompt_length:].tolist():
        # Everything after the first stop token is padding.
        end = next((k for k, t in enumerate(tokens) if t in stop_ids), None)
        n_tokens = end if end is not None else len(tokens)
        text = tokenizer.decode(tokens[:n_tokens], skip_special_tokens=True)
        stopped_after_answer = cfg["stop_after_answer"] and bool(boxed_answers(text))
        finished = end is not None or stopped_after_answer or n_tokens < cfg["max_new_tokens"]
        results.append((text, n_tokens, finished))
    return results


def run_model(model_key):
    cfg = MODELS[model_key]
    path = RESULTS_DIR / f"{model_key}.jsonl"
    done_ids = {r["id"] for r in load_results(path)}
    todo = [p for p in problems if p["id"] not in done_ids]
    print(f"{model_key}: {len(done_ids)} problems already done, {len(todo)} to go. Saving to {path}")

    if todo:
        model, tokenizer = load_model(cfg["name"])
        set_seed(SEED)
        for start in range(0, len(todo), BATCH_SIZE):
            batch = todo[start:start + BATCH_SIZE]
            t0 = time.time()
            outputs = generate_answers(model, tokenizer, [p["question"] for p in batch], cfg)
            results = []
            for p, (text, n_tokens, finished) in zip(batch, outputs):
                answer = extract_answer(text, cfg["thinks"])
                results.append({**p, "model": cfg["name"], "output": text, "answer": answer,
                                "correct": is_correct(answer, p["gold"]),
                                "n_tokens": n_tokens, "finished": finished})
            save_results(path, results)
            print(f"  saved {len(done_ids) + start + len(batch)}/{len(problems)}  ({time.time() - t0:.0f} s)")
        unload_model(model)

    # Short sanity check, so problems with the setup are noticed right away.
    results = load_results(path)
    print(f"Done: {sum(r['correct'] for r in results)}/{len(results)} correct | "
          f"{sum(not r['finished'] for r in results)} cut off at the token limit | "
          f"{sum(r['answer'] is None for r in results)} without a boxed answer")

## 4. Run model 1: DeepSeek-R1-Distill-Qwen-1.5B
The reasoning model writes long "thinking" before answering. A batch takes as long as its longest answer, so expect roughly 5 minutes per batch of 16 (about 20 minutes for 50 problems).

In [ ]:
run_model("r1-distill")

## 5. Run model 2: Qwen2.5-Math-1.5B
Same problems. The first model was removed from the GPU, so there is room for this one. This model is much faster (short answers, stopped right after its `\boxed{}` answer).

In [ ]:
run_model("qwen-math")

## 6. Look at the answers
Both models side by side, one row per problem (read back from Drive, so this also works after a disconnect).

`18 ✓` = correct, `180 ✗` = wrong, `cut off` = hit the token limit before answering, `no answer` = finished without a `\boxed{}` answer.
The answers are re-extracted from the saved outputs here, so improving `extract_answer` never requires re-running the models.

In [ ]:
def answer_cell(r, thinks):
    answer = extract_answer(r["output"], thinks)
    if answer is None:
        return "no answer" if r["finished"] else "cut off"
    return f"{answer}  {'✓' if is_correct(answer, r['gold']) else '✗'}"

comparison = pd.DataFrame(problems).set_index("id")[["gold"]]
for key, cfg in MODELS.items():
    results = {r["id"]: r for r in load_results(RESULTS_DIR / f"{key}.jsonl")}
    comparison[key] = [answer_cell(results[i], cfg["thinks"]) if i in results else "not run"
                       for i in comparison.index]

pd.set_option("display.max_rows", None)
comparison

In [ ]:
# Read one full output. Change these two values to look at other problems / the other model.
MODEL_KEY  = "r1-distill"    # "r1-distill" or "qwen-math"
PROBLEM_ID = 0

example = {r["id"]: r for r in load_results(RESULTS_DIR / f"{MODEL_KEY}.jsonl")}[PROBLEM_ID]
print("QUESTION:", example["question"])
print(f"\nCORRECT ANSWER: {example['gold']} | "
      f"MODEL ANSWER: {extract_answer(example['output'], MODELS[MODEL_KEY]['thinks'])} | "
      f"{example['n_tokens']} tokens | finished: {example['finished']}")
print("\nMODEL OUTPUT:\n", example["output"])